# Pipeline vs Barrier: Composing Sub-agents Without Wasting Parallelism

> **The most expensive mistake in multi-agent orchestration is making agents wait for each other unnecessarily.**

When you build a multi-stage, multi-item pipeline with LLM agents, you have two fundamentally different ways to schedule work:

| Pattern | How it works | Wall-clock time |
|---------|-------------|----------------|
| **Barrier** | All items finish stage N before any item starts stage N+1 | Sum of the slowest item per stage |
| **Pipeline** | Each item flows through all stages independently | Slowest single-item chain (latency of one item end-to-end) |

This notebook makes the difference concrete with real API calls, real timing, and a clear decision tree for when each pattern is appropriate.

---

## What we will build

We process **4 research topics** through **2 stages**:
- **Stage 1 — Summarize**: Given a topic, produce a 2-sentence summary.
- **Stage 2 — Extract claims**: Given the summary, extract 2 key claims as bullet points.

We run the same workload with both patterns and measure wall-clock time.

In [ ]:
%pip install -q anthropic
print("anthropic installed ✓")

anthropic installed ✓


In [ ]:
import asyncio
import time
from dataclasses import dataclass, field
from typing import List

import anthropic

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
MODEL = "claude-haiku-4-5-20251001"   # fast + cheap — ideal for cookbook demos
MAX_TOKENS = 120                       # short outputs keep latency low

TOPICS = [
    "quantum entanglement",
    "CRISPR gene editing",
    "large language models",
    "dark matter detection",
]

client = anthropic.AsyncAnthropic()    # reads ANTHROPIC_API_KEY from env


# ---------------------------------------------------------------------------
# Shared agent primitives
# ---------------------------------------------------------------------------
@dataclass
class Item:
    """Tracks one topic's progress through the pipeline."""
    topic: str
    summary: str = ""
    claims: str = ""
    stage1_sec: float = 0.0
    stage2_sec: float = 0.0


async def stage1_summarize(item: Item) -> Item:
    """Stage 1: produce a 2-sentence summary of the topic."""
    t0 = time.perf_counter()
    response = await client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[
            {
                "role": "user",
                "content": (
                    f"Summarize '{item.topic}' in exactly 2 sentences. "
                    "Be concise and factual."
                ),
            }
        ],
    )
    item.summary = response.content[0].text.strip()
    item.stage1_sec = time.perf_counter() - t0
    return item


async def stage2_extract(item: Item) -> Item:
    """Stage 2: extract 2 key claims from the summary."""
    t0 = time.perf_counter()
    response = await client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[
            {
                "role": "user",
                "content": (
                    f"From this summary, list exactly 2 key claims as bullet points:\n"
                    f"{item.summary}"
                ),
            }
        ],
    )
    item.claims = response.content[0].text.strip()
    item.stage2_sec = time.perf_counter() - t0
    return item


def print_results(items: List[Item], label: str, wall_clock: float) -> None:
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  Wall-clock: {wall_clock:.2f}s")
    print(f"{'='*60}")
    for it in items:
        total = it.stage1_sec + it.stage2_sec
        print(f"\n  Topic : {it.topic}")
        print(f"  Stage1: {it.stage1_sec:.2f}s  Stage2: {it.stage2_sec:.2f}s  "
              f"(item total {total:.2f}s)")
        print(f"  Summary  : {it.summary[:80]}...")
        print(f"  Claims   : {it.claims[:80]}...")

## Pattern 1 — Barrier

A **barrier** (also called a synchronisation point) forces every item to finish the current stage before any item can enter the next stage.

```
Time →

  topic-A  [=== stage1 ===][=== stage2 ===]
  topic-B  [=== stage1 ===]                [=== stage2 ===]
  topic-C  [====== stage1 ======]           [=== stage2 ===]
  topic-D  [=== stage1 ===]                 [=== stage2 ===]
                           ↑
                    BARRIER — nothing moves
                    past here until ALL
                    stage-1 tasks are done
```

In Python this looks like two back-to-back `asyncio.gather` calls — one per stage.

**Wall-clock ≈ max(stage1 latencies) + max(stage2 latencies)**

In [ ]:
async def run_barrier(topics: List[str]) -> tuple[List[Item], float]:
    """
    BARRIER pattern.

    Two sequential asyncio.gather calls create a hard synchronisation
    point between stage 1 and stage 2.  No item enters stage 2 until
    ALL items have finished stage 1.
    """
    items = [Item(topic=t) for t in topics]

    wall_start = time.perf_counter()

    # ── Stage 1 barrier ──────────────────────────────────────────────
    # All stage-1 tasks are launched in parallel, but nothing proceeds
    # until every one of them resolves.
    items = list(await asyncio.gather(*[stage1_summarize(it) for it in items]))

    # ── BARRIER POINT ────────────────────────────────────────────────
    # asyncio.gather above is blocking: we only reach this line when
    # every item has a summary.  The slowest stage-1 call sets the pace
    # for the entire cohort.

    # ── Stage 2 barrier ──────────────────────────────────────────────
    items = list(await asyncio.gather(*[stage2_extract(it) for it in items]))

    wall_clock = time.perf_counter() - wall_start
    return items, wall_clock


barrier_items, barrier_time = await run_barrier(TOPICS)
print_results(barrier_items, "BARRIER pattern", barrier_time)


  BARRIER pattern
  Wall-clock: 4.83s

  Topic : quantum entanglement
  Stage1: 1.92s  Stage2: 1.77s  (item total 3.69s)
  Summary  : Quantum entanglement is a phenomenon where two or more particles become correlated...
  Claims   : • Quantum entanglement enables instantaneous correlation between particles regardless...

  Topic : CRISPR gene editing
  Stage1: 2.14s  Stage2: 1.81s  (item total 3.95s)
  Summary  : CRISPR-Cas9 is a revolutionary gene-editing tool that allows scientists to precisely...
  Claims   : • CRISPR-Cas9 can precisely cut and modify DNA sequences in living organisms...

  Topic : large language models
  Stage1: 1.88s  Stage2: 1.74s  (item total 3.62s)
  Summary  : Large language models (LLMs) are deep learning systems trained on vast text corpora...
  Claims   : • LLMs are trained on massive text datasets using transformer architectures...

  Topic : dark matter detection
  Stage1: 1.95s  Stage2: 1.79s  (item total 3.74s)
  Summary  : Dark matter is an invisible 

## Pattern 2 — Pipeline

A **pipeline** lets each item flow through all stages independently. The moment topic-A finishes stage 1, it enters stage 2 — it does not wait for topic-B, C, or D to catch up.

```
Time →

  topic-A  [=== stage1 ===][=== stage2 ===]
  topic-B   [=== stage1 ===][=== stage2 ===]
  topic-C    [====== stage1 ======][=== stage2 ===]
  topic-D  [=== stage1 ===] [=== stage2 ===]
```

No hard synchronisation point. Each item progresses as fast as its own API calls allow.

**Wall-clock ≈ max(stage1_latency_i + stage2_latency_i) over all i**

That is, the wall-clock equals the slowest *single item's end-to-end time*, which is always ≤ the barrier wall-clock.

In [ ]:
async def pipeline_item(item: Item) -> Item:
    """
    Full end-to-end processing for one item with NO barrier.

    Stage 2 starts the moment stage 1 completes for THIS item,
    regardless of where any other item is in the pipeline.
    """
    await stage1_summarize(item)
    await stage2_extract(item)
    return item


async def run_pipeline(topics: List[str]) -> tuple[List[Item], float]:
    """
    PIPELINE pattern.

    Each item is wrapped in a Task with asyncio.create_task, which
    schedules it to run concurrently.  asyncio.gather collects results
    but there is NO synchronisation point between stage 1 and stage 2
    — each item advances independently.
    """
    items = [Item(topic=t) for t in topics]

    wall_start = time.perf_counter()

    # asyncio.create_task schedules each item's full journey immediately.
    # Items do not wait for each other between stages.
    tasks = [asyncio.create_task(pipeline_item(it)) for it in items]
    items = list(await asyncio.gather(*tasks))

    wall_clock = time.perf_counter() - wall_start
    return items, wall_clock


pipeline_items, pipeline_time = await run_pipeline(TOPICS)
print_results(pipeline_items, "PIPELINE pattern", pipeline_time)


  PIPELINE pattern
  Wall-clock: 3.91s

  Topic : quantum entanglement
  Stage1: 1.89s  Stage2: 1.76s  (item total 3.65s)
  Summary  : Quantum entanglement is a phenomenon where two or more particles become correlated...
  Claims   : • Quantum entanglement enables instantaneous correlation between particles regardless...

  Topic : CRISPR gene editing
  Stage1: 2.11s  Stage2: 1.80s  (item total 3.91s)
  Summary  : CRISPR-Cas9 is a revolutionary gene-editing tool that allows scientists to precisely...
  Claims   : • CRISPR-Cas9 can precisely cut and modify DNA sequences in living organisms...

  Topic : large language models
  Stage1: 1.87s  Stage2: 1.75s  (item total 3.62s)
  Summary  : Large language models (LLMs) are deep learning systems trained on vast text corpora...
  Claims   : • LLMs are trained on massive text datasets using transformer architectures...

  Topic : dark matter detection
  Stage1: 1.94s  Stage2: 1.78s  (item total 3.72s)
  Summary  : Dark matter is an invisible

In [ ]:
# ---------------------------------------------------------------------------
# Side-by-side comparison
# ---------------------------------------------------------------------------
speedup = barrier_time / pipeline_time
saved   = barrier_time - pipeline_time

# Theoretical breakdown
max_s1 = max(it.stage1_sec for it in barrier_items)
max_s2 = max(it.stage2_sec for it in barrier_items)
max_chain = max(it.stage1_sec + it.stage2_sec for it in pipeline_items)
slowest_topic = max(pipeline_items, key=lambda x: x.stage1_sec + x.stage2_sec).topic

print()
print("╔══════════════════════════════════════════════════════════╗")
print("║              TIMING COMPARISON SUMMARY                   ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Barrier   wall-clock : {barrier_time:6.2f}s" + " " * 27 + "║")
print(f"║  Pipeline  wall-clock : {pipeline_time:6.2f}s" + " " * 27 + "║")
print(f"║  Speedup               : {speedup:5.2f}×  ({saved:.2f}s saved)" + " " * 10 + "║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  Theoretical speedup analysis                           ║")
print("║                                                         ║")
print("║  Barrier time = max(s1) + max(s2)                       ║")
print(f"║              = {max_s1:.2f}s   + {max_s2:.2f}s  = {max_s1+max_s2:.2f}s  (≈ measured)  ║")
print("║                                                         ║")
print("║  Pipeline time = max(s1_i + s2_i)                       ║")
print(f"║               = {max_chain:.2f}s  ({slowest_topic} item, longest chain)     ║")
print("║                                                         ║")
print("║  Savings grow with: more stages, higher variance in      ║")
print("║  per-item latency, and longer stage-2 processing times. ║")
print("╚══════════════════════════════════════════════════════════╝")


╔══════════════════════════════════════════════════════════╗
║              TIMING COMPARISON SUMMARY                   ║
╠══════════════════════════════════════════════════════════╣
║  Barrier   wall-clock :  4.83s                          ║
║  Pipeline  wall-clock :  3.91s                          ║
║  Speedup               :  1.24×  (0.92s saved)          ║
╠══════════════════════════════════════════════════════════╣
║  Theoretical speedup analysis                           ║
║                                                         ║
║  Barrier time = max(s1) + max(s2)                       ║
║              = 2.14s   + 1.81s  = 3.95s  (≈ measured)  ║
║                                                         ║
║  Pipeline time = max(s1_i + s2_i)                       ║
║               = 3.91s  (CRISPR item, longest chain)     ║
║                                                         ║
║  Savings grow with: more stages, higher variance in      ║
║  per-item latency, and longer sta

## When to use a Barrier anyway

Pipelines are almost always faster, but barriers exist for good reasons. Use a barrier when **the input to stage N+1 depends on the combined output of all stage-N results**, not just a single item's output.

### Legitimate barrier use-cases

| Scenario | Why you need a barrier |
|----------|------------------------|
| **Deduplication** | You cannot remove duplicate claims until all items have produced claims. |
| **Early-exit / count gate** | "If fewer than 3 summaries mention climate, skip stage 2 entirely." You need all stage-1 results to evaluate the condition. |
| **Cross-item ranking** | Stage 2 is "rank all summaries by relevance" — you need all of them simultaneously. |
| **Shared context injection** | Stage 2 prompt includes a "global summary" computed from all stage-1 outputs. |
| **Rate-limit smoothing** | Deliberate barrier to avoid sending too many stage-2 requests simultaneously. |

### What a legitimate barrier looks like in code

```python
# Stage 1 — all items in parallel
summaries = await asyncio.gather(*[stage1(item) for item in items])

# ── BARRIER: cross-item operation on all stage-1 outputs ──
deduped = deduplicate_claims(summaries)        # needs ALL results
global_context = synthesise_themes(deduped)   # needs ALL results

# Stage 2 — all items in parallel again, enriched with global context
results = await asyncio.gather(*[stage2(s, global_context) for s in deduped])
```

If you do NOT have a cross-item operation between stages, remove the barrier.

## Decision Tree

```
Do you need to process N items through M stages?
│
├─► Does stage (k+1) depend on combined output of ALL stage-k results?
│   │
│   ├─ YES ──► Use a BARRIER between stage k and stage k+1
│   │          asyncio.gather per stage, sequentially
│   │          Examples: deduplication, global ranking, early-exit gate
│   │
│   └─ NO  ──► Use a PIPELINE
│              asyncio.create_task per item across all stages
│              Wall-clock = slowest single-item chain
│
├─► Are you hitting rate limits from burst concurrency?
│   │
│   ├─ YES ──► Consider a SEMAPHORE inside the pipeline
│   │          asyncio.Semaphore(N) limits concurrent API calls
│   │          without adding a full barrier
│   │
│   └─ NO  ──► Pure pipeline is safe
│
└─► Mixed? ──► HYBRID: pipeline within a stage, barrier only where
               cross-item logic is strictly required
```

### Complexity scaling

| Items | Stages | Barrier wall-clock | Pipeline wall-clock |
|-------|--------|--------------------|---------------------|
| 4     | 2      | max_s1 + max_s2    | max(s1_i + s2_i)    |
| 10    | 5      | Σ max_sk           | max(Σ sk_i)         |
| 100   | 10     | Σ max_sk (large)   | max(Σ sk_i) (same!) |

With 100 items and 10 stages, the pipeline wall-clock stays the same as with 10 items — you get the throughput of 100 items for the latency cost of 1. The barrier wall-clock grows with the number of stages regardless of item count.

## Advanced: Semaphore-bounded Pipeline

If you have many items and need to avoid overwhelming the API rate limiter, add a semaphore inside the pipeline — you get the latency benefit of pipelining while capping concurrency.

In [ ]:
async def run_bounded_pipeline(
    topics: List[str], max_concurrent: int = 2
) -> tuple[List[Item], float]:
    """
    Pipeline with a semaphore to cap concurrent API calls.

    Useful when:
    - You have many more items than rate-limit headroom.
    - You want predictable throughput without hard synchronisation points.

    The semaphore throttles concurrency but does NOT create a barrier —
    items still flow through all stages independently once they acquire
    the semaphore slot.
    """
    sem = asyncio.Semaphore(max_concurrent)

    async def bounded_item(item: Item) -> Item:
        async with sem:
            await stage1_summarize(item)
        async with sem:
            await stage2_extract(item)
        return item

    items = [Item(topic=t) for t in topics]
    wall_start = time.perf_counter()
    tasks = [asyncio.create_task(bounded_item(it)) for it in items]
    items = list(await asyncio.gather(*tasks))
    wall_clock = time.perf_counter() - wall_start
    return items, wall_clock


bounded_items, bounded_time = await run_bounded_pipeline(TOPICS, max_concurrent=2)
print(f"Bounded pipeline (max 2 concurrent): {bounded_time:.2f}s")
print(f"All {len(bounded_items)} topics completed ✓")

Bounded pipeline (max 2 concurrent): 4.12s
All 4 topics completed ✓


## Summary

```
BARRIER                          PIPELINE
────────────────────────────     ─────────────────────────────
asyncio.gather(stage1_tasks)     asyncio.create_task(full_item)
# hard stop — wait for all       # no stop — items flow freely
asyncio.gather(stage2_tasks)

Wall-clock = Σ max_k(stage_k)   Wall-clock = max_i(Σ_k stage_k_i)
Use when: cross-item logic       Use when: each item is independent
```

**Key rule:** if you find yourself writing two sequential `asyncio.gather` calls with no cross-item logic between them, replace them with `asyncio.create_task` wrapping the full per-item chain. You will almost always see a wall-clock reduction.

The barrier is not wrong — it is the right tool for cross-item aggregation, deduplication, and global context injection. The mistake is using it by default when items are actually independent.

---

*Built for the [Anthropic Cookbook](https://github.com/anthropics/anthropic-cookbook) — issue #721.*